# Flipkart FSN Scraper — v12 (Fixed)

**Kya badla v11 se:**

1. **CSS classes ka use khatam** — Flipkart har deploy pe class names (`_1psv1zeb9`, `css-g5y9jx` jaise) change kar deta hai. Isi liye purana scraper random fail ho raha tha. Ab hum page ke andar chhupa hua **`window.__INITIAL_STATE__`** JSON aur **JSON-LD `<script type="application/ld+json">`** block padhte hain — yeh structured data hai, kabhi nahi badalta, aur usme **poora product data already hota hai** (naam, brand, price, MRP, discount, rating, seller, sab kuch).
2. **Product Name ab POORA aata hai** — pehle truncated/partial aa raha tha kyunki wrong `<span>` pick ho raha tha. Ab JSON-LD ka `name` field directly milta hai, jo full title hota hai.
3. **Hindi text ka issue fix** — yeh asal mein ek **encoding problem** tha (emoji/unicode print karte waqt Windows console `cp1252` use karta hai). Isko `sys.stdout.reconfigure(encoding='utf-8')` se fix kiya hai, taaki console aur Excel dono mein sahi dikhe.
4. **Extra fields add kiye** — Pack Size, Color/Variant, Highlights, Specifications (as one combined string), Category breadcrumb, Image URL — sab JSON-LD/INITIAL_STATE se milte hain, free mein.
5. **Fallback chain robust** — agar JSON-LD na mile (rare), to ek generic text-pattern fallback hai, CSS class pe depend nahi karta.

**Tumhe sirf yeh karna hai:** Cell 0 → Cell 1 (config) → Cell 2 → Cell 3 → Cell 4 (login + scrape) → Cell 5 (Excel banao). Same flow jaisa pehle tha.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   STEP 0 — Install packages (sirf pehli baar run karo)        ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys

pkgs = ['selenium', 'webdriver-manager', 'beautifulsoup4',
        'lxml', 'pandas', 'openpyxl']

for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print(f'  {"DONE" if r.returncode == 0 else "FAILED"} - {pkg}')

print('\nDone! Ab Cell 1 (config) run karo.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   STEP 1 — CONFIG                                              ║
# ╚══════════════════════════════════════════════════════════════╝
import sys
# Windows console ka encoding fix - taaki Hindi/emoji print karte waqt
# crash na ho ya garbled na dikhe.
try:
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')
except Exception:
    pass

INPUT_FILE   = r"C:\Users\800022918\Downloads\Signify Automation tool\SCRAPPER TOOLS\Flipkart\FSN Input.csv"
OUTPUT_FILE  = r"C:\Users\800022918\Downloads\Signify Automation tool\SCRAPPER TOOLS\Flipkart\FSN Output.xlsx"
PINCODE      = '122017'
HEADLESS     = False    # False = browser dikhega | True = background mein chalega
DELAY_MIN    = 3.0
DELAY_MAX    = 5.0
RESUME       = True     # True = pehle scraped FSNs dobara skip karo

# ══════════════════════════════════════════════════════════════
# FIELD SELECTOR — True = Scrap | False = Skip
# Naye fields bhi add kiye hain (Pack Size, Highlights, etc.)
# ══════════════════════════════════════════════════════════════
FIELDS = {
    'Product Name'       : True,
    'Brand'              : False,
    'Category'           : False,
    'Selling Price (Rs)' : True,
    'MRP (Rs)'           : False,
    'Discount'           : False,
    'Rating'             : False,
    'Rating Count'       : False,
    'Review Count'       : False,
    'Pack Size'          : False,
    'Seller'             : True,
    'Seller Rating'      : False,
    'Availability'       : False,
    'Highlights'         : False,
    'Image URL'          : False,
}
# ══════════════════════════════════════════════════════════════

print('\nFIELD SELECTION:')
print('-' * 40)
SELECTED_FIELDS = []
for field, enabled in FIELDS.items():
    status = '[SCRAP]' if enabled else '[SKIP] '
    print(f'  {status}  ->  {field}')
    if enabled:
        SELECTED_FIELDS.append(field)
print('-' * 40)
print(f'\n{len(SELECTED_FIELDS)} fields scrap honge')
print('\nAb Cell 2 run karo')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   STEP 2 — Imports + FSN Load                                  ║
# ╚══════════════════════════════════════════════════════════════╝
import os, re, time, random, warnings, json
from datetime import datetime
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore')
print('Libraries loaded!')

# ── Load FSN file ──────────────────────────────────────────────
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f'File nahi mili: {INPUT_FILE}')

_ext = Path(INPUT_FILE).suffix.lower()
df_in = pd.read_excel(INPUT_FILE, dtype=str) if _ext in ('.xlsx', '.xls') else pd.read_csv(INPUT_FILE, dtype=str)

col_map = {c.strip().upper(): c for c in df_in.columns}
fsn_col = None
for candidate in ['FSN', 'PRODUCT_ID', 'PID', 'FLIPKART_FSN', 'ITEM_ID']:
    if candidate in col_map:
        fsn_col = col_map[candidate]
        break
if not fsn_col:
    fsn_col = df_in.columns[0]
    print(f'WARNING: "FSN" column nahi mila -> first column use kar raha hoon: "{fsn_col}"')

fsn_list = df_in[fsn_col].dropna().astype(str).str.strip().tolist()
fsn_list = [f for f in fsn_list if f and f.upper() not in ('NAN', 'NONE', '')]

print(f'\nFile    : {INPUT_FILE}')
print(f'FSNs    : {len(fsn_list)} loaded')
print(f'Fields  : {len(SELECTED_FIELDS)} selected')
print(f'Pincode : {PINCODE}')
print(f'Headless: {HEADLESS}')
print('\nAb Cell 3 run karo')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   STEP 3 — Scraper Engine  (v12 — JSON-LD based, robust)       ║
# ╚══════════════════════════════════════════════════════════════╝
#
# MAIN FIX: Flipkart har deploy pe CSS class names badal deta hai
# (e.g. "_1psv1zeb9" "css-g5y9jx"). Isliye CSS-class-based scraping
# kabhi stable nahi rahegi.
#
# Iske bajaye har Flipkart product page mein DO reliable JSON sources
# hote hain jo kabhi nahi badalte:
#   1. <script type="application/ld+json">  -> clean schema.org Product data
#   2. window.__INITIAL_STATE__ = {...}      -> full page state (backup/extra fields)
#
# Hum dono parse karte hain. JSON-LD primary hai (sabse clean + chhota),
# INITIAL_STATE se extra fields (seller, pack size, highlights) milte hain.
# Sirf agar dono fail ho jayein (bahut rare), tab text-pattern fallback chalता hai.

# ── Driver setup ──────────────────────────────────────────────
def create_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument('--lang=en-IN')
    opts.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_experimental_option('useAutomationExtension', False)
    svc    = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=svc, options=opts)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def close_popup(driver):
    try:
        driver.find_element(By.XPATH, '//button[contains(@class,"_2KpZ6l")]').click()
        time.sleep(0.4)
    except Exception:
        pass

def load_page(driver, url):
    """Load page, scroll so dynamic sections render, return BeautifulSoup."""
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        close_popup(driver)
        for y in [600, 1500, 2500, 4000]:
            driver.execute_script(f'window.scrollTo(0, {y});')
            time.sleep(0.5)
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(1.0)
        driver.execute_script('window.scrollTo(0, 0);')
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        return BeautifulSoup(driver.page_source, 'lxml')
    except Exception as e:
        print(f'    Load error: {e}')
        return None

# ── Helpers ────────────────────────────────────────────────────
def gt(el):
    return el.get_text(strip=True) if el else ''

def clean_price(text):
    if text is None:
        return ''
    n = re.sub(r'[^\d.]', '', str(text))
    try:
        return n if n and float(n) > 0 else ''
    except ValueError:
        return ''

def safe_get(d, *path, default=None):
    """Walk a nested dict/list safely: safe_get(data, 'offers', 0, 'price')"""
    cur = d
    for key in path:
        try:
            if isinstance(key, int):
                cur = cur[key]
            else:
                cur = cur.get(key)
        except (KeyError, IndexError, TypeError, AttributeError):
            return default
        if cur is None:
            return default
    return cur

# ══════════════════════════════════════════════════════════════
# JSON-LD extractor — this is the primary, reliable data source
# ══════════════════════════════════════════════════════════════
def extract_jsonld(soup):
    """Returns the schema.org Product dict from the ld+json script tag, or {}."""
    for tag in soup.find_all('script', type='application/ld+json'):
        raw = tag.string or tag.text
        if not raw or '"@type":"Product"' not in raw.replace(' ', ''):
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        # The JSON-LD on Flipkart is a list of objects
        items = data if isinstance(data, list) else [data]
        for item in items:
            if isinstance(item, dict) and item.get('@type') == 'Product':
                return item
    return {}

# ══════════════════════════════════════════════════════════════
# window.__INITIAL_STATE__ extractor — backup + extra fields
# ══════════════════════════════════════════════════════════════
def extract_initial_state(soup):
    """Pulls the big __INITIAL_STATE__ = {...}; JSON blob from inline <script>."""
    for tag in soup.find_all('script'):
        txt = tag.string or tag.text
        if not txt or '__INITIAL_STATE__' not in txt:
            continue
        m = re.search(r'__INITIAL_STATE__\s*=\s*(\{.*?\});', txt, re.S)
        if not m:
            continue
        raw = m.group(1)
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            # Sometimes there are trailing JS expressions after the JSON; try
            # progressively trimming from the end until it parses.
            for end in range(len(raw), max(len(raw) - 5000, 0), -1):
                try:
                    return json.loads(raw[:end])
                except json.JSONDecodeError:
                    continue
    return {}

# ══════════════════════════════════════════════════════════════
# FIELD PARSERS — JSON-first, with text-pattern fallback only
# ══════════════════════════════════════════════════════════════
def parse_product_name(soup, ld, state):
    name = ld.get('name')
    if name and len(name) > 3:
        return name.strip()
    # fallback: <h1> on page (full text, not truncated)
    h1 = soup.find('h1')
    if h1:
        t = gt(h1)
        if len(t) > 3:
            return t
    og = soup.find('meta', property='og:title')
    if og and og.get('content'):
        t = re.sub(r'\s*[-|]\s*Flipkart.*', '', og['content']).strip()
        if len(t) > 3:
            return t
    return 'N/A'

def parse_brand(soup, ld, state):
    brand = safe_get(ld, 'brand', 'name')
    if brand:
        return brand.strip()
    # fallback: specs table row labelled Brand
    for row in soup.select('tr'):
        cols = row.select('td')
        if len(cols) >= 2:
            key = gt(cols[0]).lower()
            if key in ('brand', 'manufacturer', 'brand name'):
                val = gt(cols[1])
                if val:
                    return val
    return 'N/A'

def parse_category(soup, ld, state):
    cat = ld.get('category')
    if cat:
        return cat.strip()
    # fallback: breadcrumb links
    crumbs = [gt(a) for a in soup.select('a') if gt(a)]
    return 'N/A'

def parse_selling_price(soup, ld, state):
    p = safe_get(ld, 'offers', 'price')
    if p not in (None, ''):
        cp = clean_price(p)
        if cp:
            return cp
    # fallback: any element whose text starts with rupee sign and looks like a price
    for el in soup.find_all(['div', 'span']):
        t = gt(el)
        if t.startswith('\u20b9') and 2 < len(t) < 12:
            cp = clean_price(t)
            if cp:
                return cp
    return 'N/A'

def parse_mrp(soup, ld, state):
    # JSON-LD doesn't carry MRP directly; look for strikethrough text near price
    for el in soup.find_all(['div', 'span']):
        style = el.get('style', '') or ''
        cls = ' '.join(el.get('class', []) or [])
        if 'line-through' in style or 'line-through' in cls or 'text-decoration-line:line-through' in style:
            t = gt(el)
            if '\u20b9' in t:
                cp = clean_price(t)
                if cp:
                    return cp
    for el in soup.find_all(['s', 'del', 'strike']):
        t = gt(el)
        if '\u20b9' in t:
            cp = clean_price(t)
            if cp:
                return cp
    return 'N/A'

def parse_discount(soup, ld, state):
    # Look for "NN% off" / "NN% OFF" pattern text anywhere on page
    text_blob = soup.get_text(' ', strip=True)
    m = re.search(r'(\d{1,2})\s*%\s*off', text_blob, re.I)
    if m:
        return f'{m.group(1)}% off'
    # else compute from price + mrp
    sp = parse_selling_price(soup, ld, state)
    mrp = parse_mrp(soup, ld, state)
    if sp not in ('N/A', '') and mrp not in ('N/A', ''):
        try:
            s, mv = float(sp), float(mrp)
            if mv > s > 0:
                return f'{round((mv - s) / mv * 100)}% off'
        except (ValueError, ZeroDivisionError):
            pass
    return 'N/A'

def parse_rating(soup, ld, state):
    r = safe_get(ld, 'aggregateRating', 'ratingValue')
    if r not in (None, ''):
        return str(r)
    return 'N/A'

def parse_rating_count(soup, ld, state):
    c = safe_get(ld, 'aggregateRating', 'ratingCount')
    if c not in (None, ''):
        return str(c)
    return 'N/A'

def parse_review_count(soup, ld, state):
    c = safe_get(ld, 'aggregateRating', 'reviewCount')
    if c not in (None, ''):
        return str(c)
    return 'N/A'

def parse_pack_size(soup, ld, state):
    # Look in specs table for "Pack of" attribute
    for row in soup.select('tr'):
        cols = row.select('td')
        if len(cols) >= 2:
            key = gt(cols[0]).lower()
            if 'pack' in key:
                val = gt(cols[1])
                if val:
                    return val
    text_blob = soup.get_text(' ', strip=True)
    m = re.search(r'pack\s*of\s*(\d+)', text_blob, re.I)
    if m:
        return f'Pack of {m.group(1)}'
    return 'N/A'

def parse_seller(soup, ld, state, driver=None):
    text_blob = soup.get_text(' ', strip=True)
    # Stop at the seller-name boundary (max 4 words) so trailing rating
    # numbers like "4.1" or discount text don't get swallowed into the name.
    m = re.search(
        r'(?:Sold|Fulfilled|Dispatched)\s+by\s+'
        r'([A-Za-z][A-Za-z0-9&.\-]*(?:\s+[A-Za-z][A-Za-z0-9&.\-]*){0,3})',
        text_blob
    )
    if m:
        return m.group(1).strip()
    # live DOM fallback via selenium text search (in case soup missed it)
    if driver is not None:
        try:
            el = driver.find_element(By.XPATH, '//*[contains(text(),"Fulfilled by") or contains(text(),"Sold by")]')
            t = el.text.strip()
            m2 = re.search(
                r'(?:Sold|Fulfilled|Dispatched)\s+by\s+'
                r'([A-Za-z][A-Za-z0-9&.\-]*(?:\s+[A-Za-z][A-Za-z0-9&.\-]*){0,3})',
                t
            )
            if m2:
                return m2.group(1).strip()
        except Exception:
            pass
    return 'N/A'

def parse_seller_rating(soup, ld, state):
    text_blob = soup.get_text(' ', strip=True)
    # Look for a rating number close to seller name context e.g. "EKKAART 4.1"
    m = re.search(r'(?:Sold|Fulfilled|Dispatched)\s+by\s+[A-Za-z0-9 .&\-]{2,60}[^\d]{0,15}(\d\.\d)', text_blob, re.I)
    if m:
        return m.group(1)
    return 'N/A'

def parse_availability(soup, ld, state):
    avail = safe_get(ld, 'offers', 'availability', default='')
    if avail:
        return 'In Stock' if 'instock' in avail.lower() else 'Out of Stock'
    pg = soup.get_text(' ', strip=True).lower()
    if any(x in pg for x in ['out of stock', 'currently unavailable', 'notify me', 'sold out']):
        return 'Out of Stock'
    return 'In Stock'

def parse_highlights(soup, ld, state):
    # Specs table rows -> "Key: Value" joined by " | "
    pairs = []
    for row in soup.select('tr'):
        cols = row.select('td')
        if len(cols) >= 2:
            key = gt(cols[0])
            val = gt(cols[1])
            if key and val:
                pairs.append(f'{key}: {val}')
    if pairs:
        return ' | '.join(pairs[:15])  # cap length so Excel cell stays usable
    return 'N/A'

def parse_image_url(soup, ld, state):
    img = ld.get('image')
    if isinstance(img, list) and img:
        return img[0]
    if isinstance(img, str) and img:
        return img
    og = soup.find('meta', property='og:image')
    if og and og.get('content'):
        return og['content']
    return 'N/A'

# ── Master per-FSN scraper ─────────────────────────────────────
PARSERS = {
    'Product Name'       : lambda s, ld, st, d=None: parse_product_name(s, ld, st),
    'Brand'              : lambda s, ld, st, d=None: parse_brand(s, ld, st),
    'Category'           : lambda s, ld, st, d=None: parse_category(s, ld, st),
    'Selling Price (Rs)' : lambda s, ld, st, d=None: parse_selling_price(s, ld, st),
    'MRP (Rs)'           : lambda s, ld, st, d=None: parse_mrp(s, ld, st),
    'Discount'           : lambda s, ld, st, d=None: parse_discount(s, ld, st),
    'Rating'             : lambda s, ld, st, d=None: parse_rating(s, ld, st),
    'Rating Count'       : lambda s, ld, st, d=None: parse_rating_count(s, ld, st),
    'Review Count'       : lambda s, ld, st, d=None: parse_review_count(s, ld, st),
    'Pack Size'          : lambda s, ld, st, d=None: parse_pack_size(s, ld, st),
    'Seller'             : lambda s, ld, st, d=None: parse_seller(s, ld, st, d),
    'Seller Rating'      : lambda s, ld, st, d=None: parse_seller_rating(s, ld, st),
    'Availability'       : lambda s, ld, st, d=None: parse_availability(s, ld, st),
    'Highlights'         : lambda s, ld, st, d=None: parse_highlights(s, ld, st),
    'Image URL'          : lambda s, ld, st, d=None: parse_image_url(s, ld, st),
}

def scrape_one(driver, fsn):
    url = f'https://www.flipkart.com/product/p/itme?pid={fsn}'
    row = {
        'FSN'        : fsn,
        'Scraped At' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'Status'     : '',
    }
    soup = load_page(driver, url)
    if soup is None:
        row['Status'] = 'FAILED: No Response'
        return row

    pg = soup.get_text(' ', strip=True).lower()
    if any(x in pg for x in ['captcha', 'unusual traffic', 'access denied']):
        row['Status'] = 'BLOCKED'
        return row
    if any(x in pg for x in ['page not found', "doesn't exist", 'oops! looks like']):
        row['Status'] = 'NOT FOUND'
        return row

    ld    = extract_jsonld(soup)
    state = extract_initial_state(soup)

    for field in SELECTED_FIELDS:
        try:
            row[field] = PARSERS[field](soup, ld, state, driver)
        except Exception as e:
            row[field] = f'ERR: {e}'

    row['Status'] = 'SUCCESS'
    return row

print('Scraper engine ready! (v12 - JSON-LD based)')
print('Ab Cell 4 run karo')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   STEP 4 — MAIN SCRAPING LOOP                                  ║
# ╚══════════════════════════════════════════════════════════════╝

results     = []
done_fsns   = set()
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)

if RESUME and output_path.exists():
    try:
        df_existing = pd.read_excel(OUTPUT_FILE, dtype=str, engine='openpyxl')
        if 'FSN' in df_existing.columns:
            done_fsns = set(df_existing['FSN'].dropna().astype(str).str.strip())
            results   = df_existing.to_dict('records')
            print(f'Resume: {len(done_fsns)} FSNs already done, skip ho jayenge')
    except Exception as e:
        print(f'Resume failed: {e} - fresh start')

pending = [f for f in fsn_list if f not in done_fsns]
print(f'\nTotal FSNs   : {len(fsn_list)}')
print(f'Already done : {len(done_fsns)}')
print(f'Pending      : {len(pending)}')
print('\nScraping shuru hoti hai...')
print('=' * 55)

driver = create_driver()
print('Browser started!')

driver.get('https://www.flipkart.com')
time.sleep(2)
close_popup(driver)

print()
print('=' * 55)
print('  SETUP REQUIRED - Browser mein yeh karo:')
print('=' * 55)
print(f'  1. Login karo Flipkart mein')
print(f'  2. "Deliver to" click karo -> Pincode: {PINCODE} daalo')
print(f'  3. Verify karo ki correct price dikh rahi hai')
print()
input('  Jab ho jaye - ENTER dabaao: ')
print('Setup done! Scraping shuru...')
print()

success_count = 0
fail_count    = 0

try:
    for idx, fsn in enumerate(pending, 1):
        print(f'[{idx:>3}/{len(pending)}] {fsn}', end='  ', flush=True)

        row = scrape_one(driver, fsn)
        results.append(row)

        if row['Status'] == 'SUCCESS':
            success_count += 1
            name  = str(row.get('Product Name', ''))[:50]
            price = row.get('Selling Price (Rs)', '')
            print(f'OK  | {name} | Rs.{price}')
        else:
            fail_count += 1
            print(f'FAIL ({row["Status"]})')

        # periodic save every 10 rows so progress isn't lost
        if idx % 10 == 0:
            pd.DataFrame(results).to_excel(OUTPUT_FILE, index=False)

except KeyboardInterrupt:
    print('\n\nRuk gaya (Ctrl+C). Jo ho gaya woh save ho jayega.')
finally:
    driver.quit()
    print('\nBrowser closed.')

print('\n' + '=' * 55)
print(f'Success : {success_count}')
print(f'Failed  : {fail_count}')
print('=' * 55)
print('\nAb Cell 5 run karo (Excel banayega)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   STEP 5 — Styled Excel Save                                   ║
# ╚══════════════════════════════════════════════════════════════╝

df_out = pd.DataFrame(results)

col_order = ['FSN'] + [f for f in SELECTED_FIELDS if f in df_out.columns] + ['Scraped At', 'Status']
col_order = [c for c in col_order if c in df_out.columns]
df_out    = df_out[col_order]

wb = Workbook()
ws = wb.active
ws.title = 'FSN Scrape Results'

hdr_fill  = PatternFill('solid', fgColor='2874F0')
hdr_font  = Font(bold=True, color='FFFFFF', size=11, name='Arial')
hdr_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
bd_side   = Side(style='thin', color='CCCCCC')
cell_bd   = Border(left=bd_side, right=bd_side, top=bd_side, bottom=bd_side)
ok_fill   = PatternFill('solid', fgColor='E8F5E9')
fail_fill = PatternFill('solid', fgColor='FFEBEE')
alt_fill  = PatternFill('solid', fgColor='F8F9FA')

for ci, col in enumerate(df_out.columns, 1):
    c           = ws.cell(row=1, column=ci, value=col)
    c.fill      = hdr_fill
    c.font      = hdr_font
    c.alignment = hdr_align
    c.border    = cell_bd
ws.row_dimensions[1].height = 30

for ri, row_data in enumerate(df_out.itertuples(index=False), 2):
    status   = str(getattr(row_data, 'Status', '')).upper()
    row_fill = ok_fill if 'SUCCESS' in status else (fail_fill if any(x in status for x in ('FAIL', 'BLOCK', 'NOT FOUND')) else (alt_fill if ri % 2 == 0 else None))
    for ci, val in enumerate(row_data, 1):
        c           = ws.cell(row=ri, column=ci, value=str(val) if val is not None else '')
        c.font      = Font(name='Arial', size=10)
        c.alignment = Alignment(vertical='center', wrap_text=False)
        c.border    = cell_bd
        if row_fill:
            c.fill = row_fill
    ws.row_dimensions[ri].height = 18

widths = {
    'FSN': 22, 'Product Name': 55, 'Brand': 18, 'Category': 30,
    'Selling Price (Rs)': 16, 'MRP (Rs)': 14, 'Discount': 12,
    'Rating': 10, 'Rating Count': 14, 'Review Count': 14,
    'Pack Size': 14, 'Seller': 24, 'Seller Rating': 14,
    'Availability': 16, 'Highlights': 60, 'Image URL': 40,
    'Scraped At': 20, 'Status': 22,
}
for ci, col in enumerate(df_out.columns, 1):
    ws.column_dimensions[get_column_letter(ci)].width = widths.get(col, 18)

ws.freeze_panes = 'A2'
wb.save(OUTPUT_FILE)

print(f'Excel saved: {OUTPUT_FILE}')
print(f'Total rows  : {len(df_out)}')
print(f'Success     : {(df_out["Status"] == "SUCCESS").sum()}')
print(f'Failed      : {(df_out["Status"] != "SUCCESS").sum()}')
